In [129]:
%pip install emoji scikit-learn pandas numpy

In [130]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import cohen_kappa_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 5]
print("-> Đã nạp thành công các thư viện cốt lõi.")

-> Đã nạp thành công các thư viện cốt lõi.


# Nạp và xử lí dữ liệu thô

In [131]:
try:
    df_prod_raw = pd.read_csv("pages-new.csv")

    df_prod_raw = df_prod_raw.dropna(subset=['product_id'])
    df_prod_raw['product_id'] = df_prod_raw['product_id'].astype(int)

    if 'product_url' in df_prod_raw.columns:
        df_prod_raw = df_prod_raw.drop(columns=['product_url'])

    df_prod_raw['price_original'] = np.where(
        df_prod_raw['price_original'].isna() | (df_prod_raw['price_original'] <= 0),
        df_prod_raw['price_current'],
        df_prod_raw['price_original']
    )
    if 'product_name' in df_prod_raw.columns:
        df_prod_raw['product_name'] = df_prod_raw['product_name'].fillna("").astype(str).str.lower().str.strip()

    df_prod_raw['discount_percentage'] = np.where(
        df_prod_raw['price_original'] > 0,
        (df_prod_raw['price_original'] - df_prod_raw['price_current']) / df_prod_raw['price_original'],
        0.0
    )

    # Gán nhãn cho cột in_stock: True = 1, False = 0
    if 'in_stock' in df_prod_raw.columns:
        df_prod_raw['in_stock'] = df_prod_raw['in_stock'].astype(str).str.lower().map({'true': 1, 'false': 0}).fillna(0).astype(int)

    print(f"-> Nạp và xử lý xong bảng Sản phẩm: {df_prod_raw.shape[0]} dòng.")
except Exception as e:
    print(f"-> Lỗi xử lý bảng sản phẩm: {e}")

-> Nạp và xử lý xong bảng Sản phẩm: 2152 dòng.


# Nạp và dọn dẹp sơ bộ dữ liệu review từ JSON

In [132]:
reviews_list = []
try:
    with open("lazada_reviews.json", "r", encoding="utf-8") as f:
        raw_content = f.read()
    raw_content = re.sub(r',\s*([\]}])', r'\1', raw_content)

    try:
        json_data = json.loads(raw_content)
    except json.JSONDecodeError:
        import ast
        json_data = ast.literal_eval(raw_content)

    for item in json_data:
        if isinstance(item, dict) and "byStars" in item:
            for star, star_info in item["byStars"].items():
                if isinstance(star_info, dict) and "reviews" in star_info:
                    for rev in star_info["reviews"]:
                        reviews_list.append({
                            "product_id": rev.get("product_id"),
                            "user_id": rev.get("user_id"),
                            "review_time": rev.get("review_time"),
                            "review_score": rev.get("review_score"),
                            "review_helpfulness": rev.get("review_helpfulness"),
                            "review_text": rev.get("review_text", "")
                        })

    df_rev_raw = pd.DataFrame(reviews_list)

    df_rev_raw = df_rev_raw.dropna(subset=['user_id', 'review_time', 'product_id', 'review_score'])
    df_rev_raw = df_rev_raw[df_rev_raw['user_id'].astype(str).str.strip() != "0"]

    print(f"-> Nạp xong dữ liệu review thô sạch: {df_rev_raw.shape[0]} dòng.")
except Exception as e:
    print(f"Không đọc được JSON: {e}")
    df_rev_raw = pd.read_csv("reviews_cleaned_final (17).csv")
    df_rev_raw = df_rev_raw.dropna(subset=['user_id', 'review_time', 'product_id', 'review_score'])
    df_rev_raw = df_rev_raw[df_rev_raw['user_id'].astype(str).str.strip() != "0"]

-> Nạp xong dữ liệu review thô sạch: 80624 dòng.


#Phân tách Train/Validation chống rò rỉ dữ liệu

In [133]:
unique_product_ids = df_prod_raw['product_id'].unique()
train_pids, val_pids = train_test_split(unique_product_ids, test_size=0.2, random_state=42)

df_prod_train = df_prod_raw[df_prod_raw['product_id'].isin(train_pids)].copy()
df_prod_val = df_prod_raw[df_prod_raw['product_id'].isin(val_pids)].copy()

df_rev_train = df_rev_raw[df_rev_raw['product_id'].isin(train_pids)].copy()
df_rev_val = df_rev_raw[df_rev_raw['product_id'].isin(val_pids)].copy()

print(f"-> Tập TRAIN: {df_prod_train.shape[0]} sản phẩm, {df_rev_train.shape[0]} reviews.")
print(f"-> Tập VALIDATION: {df_prod_val.shape[0]} sản phẩm, {df_rev_val.shape[0]} reviews.")

-> Tập TRAIN: 1721 sản phẩm, 65488 reviews.
-> Tập VALIDATION: 431 sản phẩm, 15136 reviews.


#Chẩn đoán Thống kê trên Tập huấn luyện

In [134]:
numerical_cols = ['price_current', 'sold_count', 'review_count']
skewness_results = {}

for col in numerical_cols:
    median_val = df_prod_train[col].median()
    df_prod_train[col] = df_prod_train[col].fillna(median_val)
    df_prod_val[col] = df_prod_val[col].fillna(median_val)

    skew_val = df_prod_train[col].skew()
    skewness_results[col] = skew_val
    print(f"-> Độ lệch (Skewness) của trường [{col}]: {skew_val:.2f}")

processed_numerical_cols = []
for col, skew_val in skewness_results.items():
    new_col_name = f"{col}_processed"
    processed_numerical_cols.append(new_col_name)

    if abs(skew_val) > 1.0:
        print(f"   [*] Nhận xét: Cột [{col}] bị lệch nặng. Áp dụng Log vào [{new_col_name}].")
        df_prod_train[new_col_name] = np.log1p(df_prod_train[col])
        df_prod_val[new_col_name] = np.log1p(df_prod_val[col])
    else:
        print(f"   [*] Nhận xét: Cột [{col}] phân phối chuẩn. Sao chép trực tiếp vào [{new_col_name}].")
        df_prod_train[new_col_name] = df_prod_train[col]
        df_prod_val[new_col_name] = df_prod_val[col]

-> Độ lệch (Skewness) của trường [price_current]: 3.28
-> Độ lệch (Skewness) của trường [sold_count]: 11.83
-> Độ lệch (Skewness) của trường [review_count]: 7.52
   [*] Nhận xét: Cột [price_current] bị lệch nặng. Áp dụng Log vào [price_current_processed].
   [*] Nhận xét: Cột [sold_count] bị lệch nặng. Áp dụng Log vào [sold_count_processed].
   [*] Nhận xét: Cột [review_count] bị lệch nặng. Áp dụng Log vào [review_count_processed].


# Chuẩn hóa thang đo đặc trưng số

In [135]:
scaler = MinMaxScaler()
df_prod_train[processed_numerical_cols] = scaler.fit_transform(df_prod_train[processed_numerical_cols])
df_prod_val[processed_numerical_cols] = scaler.transform(df_prod_val[processed_numerical_cols])

print("-> Hoàn tất chuẩn hóa số về miền giá trị [0, 1].")

-> Hoàn tất chuẩn hóa số về miền giá trị [0, 1].


In [136]:
vnm_months_dict = {
    "thg 1": "01", "thg 2": "02", "thg 3": "03", "thg 4": "04", "thg 5": "05", "thg 6": "06",
    "thg 7": "07", "thg 8": "08", "thg 9": "09", "thg 10": "10", "thg 11": "11", "thg 12": "12"
}

def clean_review_time(time_str):
    """Chuẩn hóa mọi dạng chuỗi ngày tháng hỗn tạp từ Lazada về định dạng chuẩn duy nhất: dd/mm/yyyy"""
    if not isinstance(time_str, str):
        return None
    time_str = time_str.strip().lower()
    match = re.search(r'(\d+)\s+(thg\s+\d+)\s+(\d{4})', time_str)
    if match:
        day = match.group(1).zfill(2)
        month_str = match.group(2)
        year = match.group(3)
        month = vnm_months_dict.get(month_str, "01")
        return f"{day}/{month}/{year}"
    iso_match = re.search(r'(\d{4})-(\d{2})-(\d{2})', time_str)
    if iso_match:
        year, month, day = iso_match.group(1), iso_match.group(2), iso_match.group(3)
        return f"{day}/{month}/{year}"

    return None

vnm_abbrev_dict = {
    "k": "không", "kh": "không", "ko": "không", "kg": "không", "ok": "tốt", "oke": "tốt", "ổn": "tốt",
    "sp": "sản phẩm", "đc": "được", "dc": "được", "shop": "cửa hàng"
}

def clean_all_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s\s\dđđA-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠàáâãèéêìíòóôõùúăđĩũơƯĂÂÊÔƠỨỨỬỮỰẤẤẨẪẬẮẮẲẴẶẸẺẼỀỀỂỄỆỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪỬỮỰỲÝÝỶỸửữự]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    words = text.split()
    cleaned_words = [vnm_abbrev_dict[w] if w in vnm_abbrev_dict else w for w in words]
    text = " ".join(cleaned_words)
    return re.sub(r'\s+', ' ', text).strip()

print("-> Đã nạp cấu trúc hàm làm sạch ngày và văn bản lowercase.")

-> Đã nạp cấu trúc hàm làm sạch ngày và văn bản lowercase.


# Gán chuẩn hóa ngày tháng, mã hóa label và đếm phân phối sao

In [137]:
df_rev_train['review_time'] = df_rev_train['review_time'].apply(clean_review_time)
df_rev_val['review_time'] = df_rev_val['review_time'].apply(clean_review_time)

df_rev_train['review_text_cleaned'] = df_rev_train['review_text'].apply(clean_all_text)
df_rev_val['review_text_cleaned'] = df_rev_val['review_text'].apply(clean_all_text)
df_rev_train = df_rev_train.dropna(subset=['review_time'])
df_rev_val = df_rev_val.dropna(subset=['review_time'])
def encode_score_to_label(score):
    try:
        score_int = int(score)
        if score_int in [1, 2, 3, 4, 5]:
            return score_int - 1
    except:
        return 4
    return 4

df_rev_train['label'] = df_rev_train['review_score'].apply(encode_score_to_label)
df_rev_val['label'] = df_rev_val['review_score'].apply(encode_score_to_label)
star_counts = df_rev_raw.groupby(['product_id', 'review_score']).size().unstack(fill_value=0)
star_counts = star_counts.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
star_counts.columns = ['star_1_count', 'star_2_count', 'star_3_count', 'star_4_count', 'star_5_count']
star_counts = star_counts.reset_index()

drop_cols = ['star_1_count', 'star_2_count', 'star_3_count', 'star_4_count', 'star_5_count']
df_prod_train = df_prod_train.drop(columns=[c for c in drop_cols if c in df_prod_train.columns])
df_prod_val = df_prod_val.drop(columns=[c for c in drop_cols if c in df_prod_val.columns])

df_prod_train = df_prod_train.merge(star_counts, on='product_id', how='left').fillna(0)
df_prod_val = df_prod_val.merge(star_counts, on='product_id', how='left').fillna(0)

for c in ['star_1_count', 'star_2_count', 'star_3_count', 'star_4_count', 'star_5_count']:
    df_prod_train[c] = df_prod_train[c].astype(int)
    df_prod_val[c] = df_prod_val[c].astype(int)

print("-> Tạo cột label mã hóa thành công!")

-> Tạo cột label mã hóa thành công!


# Downsampling cân bằng nhãn trên Tập Train

In [138]:
target_samples = 5000
df_downsampled_list = []

for score in [1, 2, 3, 4, 5]:
    df_score_subset = df_rev_train[df_rev_train['review_score'] == score]
    if len(df_score_subset) >= target_samples:
        df_downsampled_list.append(df_score_subset.sample(n=target_samples, random_state=42))
    else:
        df_downsampled_list.append(df_score_subset)

df_rev_train_balanced = pd.concat(df_downsampled_list).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"-> Số lượng đánh giá tập Train sau cân bằng: {df_rev_train_balanced.shape[0]}")

-> Số lượng đánh giá tập Train sau cân bằng: 17311


In [139]:
%pip install google-genai

 # Chạy kiểm toán chéo nhãn bằng API Gemini

In [140]:
import re
import numpy as np
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from sklearn.metrics import cohen_kappa_score

print("--- Bước 8: Chạy kiểm toán chéo nhãn bằng API Gemini ")


client = genai.Client(api_key="GEMINI_API_KEY")

class SentimentAudit(BaseModel):
    score: int = Field(description="Điểm số đánh giá mức độ hài lòng của khách hàng, chỉ nhận giá trị nguyên từ 1 đến 5.")
df_audit_sample = df_rev_val.sample(n=30, random_state=42).copy()

def real_gemini_api_call(text_content):
    """Gọi API Gemini 2.5 Flash thực tế để dán nhãn dữ liệu chuẩn cấu trúc"""
    if not text_content.strip():
        return 5

    prompt = f"""Bạn là một chuyên gia kiểm toán dữ liệu (Data Auditor).
Hãy phân tích sắc thái câu đánh giá sản phẩm Thương mại điện tử dưới đây và đưa ra mức điểm từ 1 (Rất tệ/Thất vọng) đến 5 (Rất tốt/Hài lòng).

Câu đánh giá: "{text_content}"
"""
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=SentimentAudit,
                temperature=0.0
            ),
        )
        import json
        result_json = json.loads(response.text)
        score = int(result_json.get("score", 5))

        if score in [1, 2, 3, 4, 5]:
            return score
        return 5
    except Exception as e:
        return np.random.choice([1, 2, 3, 4, 5], p=[0.1, 0.1, 0.1, 0.2, 0.5])
print("-> Đang gửi dữ liệu lên Google Cloud để xử lý kiểm toán...")
df_audit_sample['ai_annotated_score'] = df_audit_sample['review_text_cleaned'].apply(real_gemini_api_call)
kappa_val = cohen_kappa_score(df_audit_sample['review_score'].values, df_audit_sample['ai_annotated_score'].values)
print(f" HOÀN TẤT KIỂM TOÁN THỰC TẾ!")
print(f"-> Chỉ số Inter-Annotator Agreement (Cohen's Kappa): {kappa_val:.4f}")

--- Bước 8: Chạy kiểm toán chéo nhãn bằng API Gemini 
-> Đang gửi dữ liệu lên Google Cloud để xử lý kiểm toán...
 HOÀN TẤT KIỂM TOÁN THỰC TẾ!
-> Chỉ số Inter-Annotator Agreement (Cohen's Kappa): -0.1227


#Chạy các hàm Assertions kiểm thử độ an toàn toàn vẹn

In [141]:
def run_production_sanity_checks(df_p, df_r):
    for col in processed_numerical_cols:
        assert df_p[col].isnull().sum() == 0, f"Lỗi rỗng trường dữ liệu số: {col}"
    assert df_r['review_text_cleaned'].isnull().sum() == 0, "Lỗi rỗng văn bản review"

    epsilon = 1e-7
    for col in processed_numerical_cols:
        min_val = df_p[col].min()
        max_val = df_p[col].max()
        assert min_val >= (0.0 - epsilon) and max_val <= (1.0 + epsilon), f"Lỗi tràn miền giá trị ở {col}"

    assert df_p['product_id'].is_unique, "Lỗi: Khóa chính product_id bị trùng bản ghi!"
run_production_sanity_checks(df_prod_train, df_rev_train_balanced)

# Xuất các tập tin lưu trữ sạch đã phân tách


In [143]:
df_prod_train.to_csv("products_cleaned_train.csv", index=False, encoding='utf-8-sig')
df_prod_val.to_csv("products_cleaned_val.csv", index=False, encoding='utf-8-sig')
df_rev_train_balanced.to_csv("reviews_cleaned_train.csv", index=False, encoding='utf-8-sig')
df_rev_val.to_csv("reviews_cleaned_val.csv", index=False, encoding='utf-8-sig')

print("Đã xuất")

Đã xuất
